## 1. Setup & Imports
All third-party imports, device selection, and global random seed.

In [ ]:
import os, math, random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 2. Configuration
All hyperparameters and paths in one place — change here only.

In [ ]:
cfg = SimpleNamespace(
    # --- Paths ---
    dataset_dir="/kaggle/input/competitions/csiro-biomass",
    dino_weights_dir="/kaggle/input/datasets/darealvictorslorer/dinov2-small-weights/dinov2-small",
    output_dir="/kaggle/working",
    # --- Model ---
    input_dim=384,
    latent_dim=32,
    output_dim=5,
    dropout=0.1,
    # --- CEMS ---
    sigma=1e-3,        # tangent perturbation std (paper default)
    cems_method=1,     # batch-centred SVD
    initial_d=5,       # intrinsic dimension estimate (from local Part-2 run: d≈4.71)
    use_hessian=False, # False → CEMS-L (first-order/linear); True → full CEMS
    # --- Training ---
    epochs=80,
    lr=3e-4,
    weight_decay=1e-3,
    seed=42,
    # --- Split ---
    n_splits=5,
    val_fold=0,
)

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1,
           "GDM_g": 0.2, "Dry_Total_g": 0.5}

TRAIN_CSV    = os.path.join(cfg.dataset_dir, "train.csv")
TEST_CSV     = os.path.join(cfg.dataset_dir, "test.csv")
TRAIN_IMG_DIR = os.path.join(cfg.dataset_dir, "train")
TEST_IMG_DIR  = os.path.join(cfg.dataset_dir, "test")

print("train.csv:",  os.path.exists(TRAIN_CSV))
print("test.csv:",   os.path.exists(TEST_CSV))
print("train dir:",  os.path.exists(TRAIN_IMG_DIR))
print("test dir:",   os.path.exists(TEST_IMG_DIR))
print("dino dir:",   os.path.exists(cfg.dino_weights_dir))

## 3. Load DINOv2
Load ViT-S/14 from uploaded HuggingFace weights; smoke-test at 504×252 resolution.

In [ ]:
print(f"Loading DINOv2 ViT-S/14 from {cfg.dino_weights_dir} ...")
dino = AutoModel.from_pretrained(cfg.dino_weights_dir).eval().to(device)
for p in dino.parameters():
    p.requires_grad_(False)

# Smoke test: 504x252 input (W=504, H=252) — must pass interpolate_pos_encoding=True
# because this is a non-standard resolution for the HF wrapper.
_dummy = torch.zeros(1, 3, 252, 504, device=device)  # (B, C, H, W)
with torch.no_grad():
    _out = dino(pixel_values=_dummy, interpolate_pos_encoding=True)
    _cls = _out.last_hidden_state[:, 0, :]  # CLS token
assert _cls.shape == (1, 384), f"Expected (1, 384), got {_cls.shape}"
print(f"DINOv2 smoke test passed — CLS shape: {_cls.shape}")
del _dummy, _out, _cls

## 4. Load Training CSV
Pivot long-format CSV to wide format (one row per image) and extract Y matrix.

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["image_id"] = df_raw["sample_id"].str.split("__").str[0]

df_wide = df_raw.pivot_table(
    index=["image_id", "image_path"],
    columns="target_name",
    values="target",
).reset_index()

Y_all = df_wide[TARGETS].values.astype(np.float32)  # (N, 5)
train_image_ids_all = df_wide["image_id"].values

print(f"Training images: {len(df_wide)}")
print(f"Y_all shape:     {Y_all.shape}")
print(df_wide.head(3))

## 5. Extract DINOv2 Features — Training Images
Preprocess at 504×252 (2:1 aspect, matching local extract_features.py) and extract CLS tokens.

In [ ]:
# 504x252 = (W x H); torchvision Resize takes (H, W) = (252, 504)
# Matches extract_features.py: DINOV2_RESIZE=(504,252), Resize((252,504))
_dino_transform = T.Compose([
    T.Resize((252, 504)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def extract_features(image_paths, model, transform):
    """Return (N, 384) float32 array of CLS-token features."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(pixel_values=x, interpolate_pos_encoding=True)
            feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
        feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


# 4-orientation augmentation: flips applied on PIL image BEFORE resize+normalise
_ORIENTATIONS = [
    lambda img: img,                      # identity
    lambda img: TF.hflip(img),            # horizontal flip
    lambda img: TF.vflip(img),            # vertical flip
    lambda img: TF.hflip(TF.vflip(img)),  # 180° rotation (= hflip + vflip)
]


def extract_features_augmented(image_paths, model, transform):
    """Return (N*4, 384) array. Each image appears 4 times (id, hflip, vflip, hflip+vflip).
    Flips are applied to the original PIL image before resize/normalise."""
    feats = []
    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB")
        for flip_fn in _ORIENTATIONS:
            x = transform(flip_fn(img)).unsqueeze(0).to(device)
            with torch.no_grad():
                out = model(pixel_values=x, interpolate_pos_encoding=True)
                feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
            feats.append(feat)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


train_image_paths = [
    os.path.join(TRAIN_IMG_DIR, f"{iid}.jpg")
    for iid in train_image_ids_all
]
N_orig = len(train_image_paths)
print(f"Extracting features for {N_orig} training images (4 orientations each)...")
X_all = extract_features_augmented(train_image_paths, dino, _dino_transform)

# Expand Y and image IDs to match augmented feature array
Y_all               = np.repeat(Y_all,               4, axis=0)  # (N*4, 5)
train_image_ids_all = np.repeat(train_image_ids_all, 4)           # (N*4,)
image_group_ids     = np.repeat(np.arange(N_orig),   4)           # (N*4,) — group = original image index

print(f"X_all shape:         {X_all.shape}")           # (1428, 384)
print(f"Y_all shape:         {Y_all.shape}")           # (1428, 5)
print(f"image_group_ids len: {len(image_group_ids)}")  # 1428
print(f"Unique groups:       {len(np.unique(image_group_ids))}")  # 357

# Sanity checks
assert X_all.shape == (N_orig * 4, 384), f"Expected ({N_orig*4}, 384), got {X_all.shape}"
assert Y_all.shape == (N_orig * 4, 5)
assert len(image_group_ids) == N_orig * 4
assert len(np.unique(image_group_ids)) == N_orig
assert np.all(np.bincount(image_group_ids) == 4), "Each group must appear exactly 4 times"
assert np.all(Y_all[0] == Y_all[1]) and np.all(Y_all[1] == Y_all[2]) and np.all(Y_all[2] == Y_all[3]), \
    "Y values for the first image's 4 orientations must be identical"
print("Sanity checks passed.")

## 6. Extract DINOv2 Features — Test Images
Same preprocessing pipeline applied to the held-out test set.

In [ ]:
df_test_raw = pd.read_csv(TEST_CSV)
df_test_raw["image_id"] = df_test_raw["sample_id"].str.split("__").str[0]
df_test_unique = df_test_raw.drop_duplicates("image_id").copy()

test_image_ids = df_test_unique["image_id"].values
test_image_paths = [
    os.path.join(TEST_IMG_DIR, f"{iid}.jpg")
    for iid in test_image_ids
]

print(f"Extracting features for {len(test_image_paths)} test images...")
X_test = extract_features(test_image_paths, dino, _dino_transform)
print(f"X_test shape: {X_test.shape}")

## 7. GroupKFold Train/Val Split
Fold 0 of a 5-fold GroupKFold; groups on image_path (one unique group per image).

In [ ]:
gkf = GroupKFold(n_splits=cfg.n_splits)
# Group by original image index so all 4 orientations of an image land in the
# same fold — prevents leakage (original in train, hflip in val = cheating).
splits = list(gkf.split(X_all, groups=image_group_ids))
train_idx, val_idx = splits[cfg.val_fold]

X_train     = X_all[train_idx]
Y_train_raw = Y_all[train_idx]
X_val       = X_all[val_idx]
Y_val_raw   = Y_all[val_idx]
train_ids_split = train_image_ids_all[train_idx]
val_ids_split   = train_image_ids_all[val_idx]

# Sanity check: no original image shared across train and val
train_groups = set(image_group_ids[train_idx].tolist())
val_groups   = set(image_group_ids[val_idx].tolist())
assert train_groups.isdisjoint(val_groups), "Group leakage: original images shared across folds!"
print("Group overlap check: PASSED (zero overlap)")

print(f"Train: {X_train.shape}  Val: {X_val.shape}")
print(f"Y_train range: [{Y_train_raw.min():.1f}, {Y_train_raw.max():.1f}]")

## 8. MinMaxScaler on Y_train
Fit scaler on training labels only; used inside CEMS joint space for label normalisation.

In [ ]:
y_scaler = MinMaxScaler()
y_scaler.fit(Y_train_raw)
Y_train_scaled = y_scaler.transform(Y_train_raw).astype(np.float32)

print(f"Y_train_scaled range: [{Y_train_scaled.min():.3f}, {Y_train_scaled.max():.3f}]")
print("Scaler data_min_:", y_scaler.data_min_.round(2))
print("Scaler data_max_:", y_scaler.data_max_.round(2))

## 9. Model Architecture — Encoder, Head, BiomassModel
Encoder 384→128→32, Head 32→32→5; forward_cems inserts CEMS augmentation between them.

In [ ]:
class Encoder(nn.Module):
    """384 → 128 → 32 (GELU, Dropout)."""

    def __init__(self, input_dim=384, latent_dim=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class Head(nn.Module):
    """32 → 32 → 5 (GELU, Dropout)."""

    def __init__(self, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class BiomassModel(nn.Module):

    def __init__(self, input_dim=384, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim, dropout)
        self.head = Head(latent_dim, output_dim, dropout)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.head(self.encode(x))

    def forward_cems(self, args, x, y_scaled):
        """CEMS augmentation path.

        Encodes x, detaches latent so SVD/ridge don't backprop through encoder,
        then runs get_batch_cems in 32-d latent space.
        Returns (pred_aug, y_aug_scaled).
        """
        latent = self.encode(x)
        latent_aug, y_aug_scaled = get_batch_cems(
            args, latent.detach(), y_scaled, latent=True,
        )
        return self.head(latent_aug), y_aug_scaled


print("BiomassModel defined.")
# Quick parameter count
_tmp = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout)
n_params = sum(p.numel() for p in _tmp.parameters())
print(f"Parameters: {n_params:,}")
del _tmp

## 10. CEMS Algorithm
Port of cems.py: _adjust_dims, _get_projection, _estimate_grad_hessian, _sample_tangent, get_batch_cems.

In [ ]:
# ---------------------------------------------------------------------------
# Per-batch intrinsic dimension estimate (TwoNN, lightweight)
# ---------------------------------------------------------------------------

def _twonn_batch(zi):
    """TwoNN ID estimate on a small batch (b, m). Returns rounded int."""
    with torch.no_grad():
        dists = torch.cdist(zi.float(), zi.float())
        dists.fill_diagonal_(float('inf'))
        top2 = dists.topk(2, dim=1, largest=False).values
        r1, r2 = top2[:, 0], top2[:, 1]
        mask = r1 > 0
        if mask.sum() < 3:
            return 2
        mu = (r2 / r1)[mask]
        mu_sorted, _ = mu.sort()
        n = len(mu_sorted)
        ecdf = torch.arange(1, n + 1, dtype=torch.float32, device=zi.device) / n
        log_mu   = torch.log(mu_sorted)
        log_surv = -torch.log1p(-ecdf + 1e-10)
        denom = (log_mu * log_mu).sum()
        if denom.abs() < 1e-12:
            return 2
        d = float((log_mu * log_surv).sum() / denom)
    if not math.isfinite(d) or d < 1:
        return 2
    return max(1, int(round(d)))


# ---------------------------------------------------------------------------
# Step 1 — flatten + form joint space zi = [x | y]
# ---------------------------------------------------------------------------

def _adjust_dims(x, y, xk=None, yk=None):
    if x.ndim > 2:
        x = x.reshape(x.shape[0], -1)
    if y.ndim == 1:
        y = y.reshape(y.shape[0], 1)
    m  = x.shape[-1]
    zi = torch.cat((x, y), dim=-1)
    zk = None
    if xk is not None and yk is not None:
        if xk.ndim > 3:
            xk = xk.reshape(xk.shape[0], xk.shape[1], -1)
        zk = torch.cat((xk, yk), dim=-1)
    return x, zi, zk, m


# ---------------------------------------------------------------------------
# Step 2 — SVD local orthonormal basis
# ---------------------------------------------------------------------------

def _get_projection(args, x, xk):
    x_c = x.transpose(-2, -1)          # (D, b)
    x_c_mean = None

    if args.cems_method == 1:
        x_c_mean = torch.mean(x_c, -1)           # (D,)
        x_c      = x_c - x_c_mean.unsqueeze(-1)  # singly-centred
    else:
        assert xk is not None
        xk_t = xk.transpose(-1, -2)
        x    = x.unsqueeze(-1)
        x_c  = xk_t - x

    svd_input  = (x_c - x_c_mean.unsqueeze(-1) if x_c_mean is not None else x_c)
    svd_kwargs = {"driver": "gesvd"} if svd_input.is_cuda else {}
    basis, _, _ = torch.linalg.svd(svd_input, full_matrices=False, **svd_kwargs)

    u      = basis.transpose(-2, -1) @ x_c  # (K, b)
    u_prev = u.transpose(-2, -1)             # (b, K)

    if args.cems_method == 1:
        u_t  = u.transpose(-1, -2)                        # (b, K)
        u    = (u_t.unsqueeze(1) - u_t).transpose(-1, -2) # (b, K, b)
        n    = x.shape[0]
        mask = ~torch.eye(n, dtype=torch.bool, device=x.device)
        u    = -u.transpose(-1, -2)[mask].reshape(
            (u.shape[0], u.shape[2] - 1, u.shape[1])
        ).transpose(-1, -2)  # (b, K, b-1)
    elif args.cems_method == 2:
        u = u.unsqueeze(0)

    return basis, u, u_prev, x_c_mean


# ---------------------------------------------------------------------------
# Ridge regression solver
# ---------------------------------------------------------------------------

def _solve_ridge(a, b, lam=1.0):
    n   = a.shape[-1]
    eye = torch.eye(n, device=a.device, dtype=a.dtype)
    a_t = a.transpose(-2, -1)
    return torch.linalg.inv(a_t @ a + lam * eye) @ a_t @ b


# ---------------------------------------------------------------------------
# Steps 3-4 — local gradient and Hessian via ridge regression
# ---------------------------------------------------------------------------

def _estimate_grad_hessian(args, x, xk, d):
    tidx      = torch.triu_indices(d, d, device=x.device)
    ones_mult = torch.ones((d, d), device=x.device)
    ones_mult.fill_diagonal_(0.5)

    basis, u, u_prev, x_mean = _get_projection(args, x, xk)

    u_d = u[:, :d]                          # (b, d, b-1)
    f   = u[:, d:].transpose(-2, -1)        # (b, b-1, n_normal)

    if args.use_hessian:
        uu = torch.einsum(
            'bki,bkj->bkij',
            u_d.transpose(-2, -1),
            u_d.transpose(-2, -1),
        )
        uu   = uu * ones_mult
        uu   = uu[:, :, tidx[0], tidx[1]].transpose(-2, -1)
        psi  = torch.cat((u_d, uu), dim=1).transpose(-2, -1)
    else:
        psi = u_d.transpose(-2, -1)  # CEMS-L: linear only

    lam    = torch.linalg.norm(psi, dim=(-1, -2)).mean()
    b_coef = _solve_ridge(psi, f, lam=lam).transpose(-2, -1)  # (b, n_normal, d or d+n_quad)

    gradient = b_coef[..., :d]
    hessian  = torch.zeros(
        (u.shape[0], b_coef.shape[1], d, d), dtype=b_coef.dtype, device=b_coef.device
    )
    if args.use_hessian:
        hessian[..., tidx[0], tidx[1]] = b_coef[..., d:]
        hessian[..., tidx[1], tidx[0]] = b_coef[..., d:]

    return basis, gradient, hessian, u_d, u_prev, x_mean


# ---------------------------------------------------------------------------
# Step 5 — sample in tangent bundle, project back to ambient space
# ---------------------------------------------------------------------------

def _sample_tangent(args, x, u_k_d, u_prev, x_mean, basis, grad, hess):
    d  = grad.shape[-1]
    nu = torch.distributions.Normal(0, args.sigma).sample(
        (x.shape[0], d, 1)
    ).to(x.device)

    f_nu = (grad @ nu).squeeze(-1)

    if args.use_hessian:
        nu_ex = nu.unsqueeze(1)
        f_nu  = f_nu + 0.5 * (nu_ex.transpose(-1, -2) @ hess @ nu_ex).squeeze((-1, -2))

    x_new_local = torch.cat((nu.squeeze(-1), f_nu), dim=-1)

    if args.cems_method == 1:
        x_new_local = x_new_local + u_prev

    x_cems = (basis @ x_new_local.unsqueeze(-1)).squeeze(-1)
    x_cems = x_cems + (x_mean if args.cems_method == 1 else x)
    return x_cems


# ---------------------------------------------------------------------------
# Public entry — get_batch_cems
# ---------------------------------------------------------------------------

def get_batch_cems(args, x, y, xk=None, yk=None, *, latent=True):
    """CEMS augmentation for one minibatch. Returns (x_new, y_new)."""
    x_shape, y_shape = x.shape, y.shape
    x_flat, zi, zk, m = _adjust_dims(x, y, xk, yk)

    d = args.id
    if latent:
        with torch.no_grad():
            d = _twonn_batch(zi)

    d = min(d, zi.shape[-1] - 1, zi.shape[-2] - 1)
    d = max(d, 1)

    basis, grad, hess, u_k_d, u_prev, x_mean = _estimate_grad_hessian(args, zi, zk, d)
    z_sampled = _sample_tangent(args, zi, u_k_d, u_prev, x_mean, basis, grad, hess)

    x_new = z_sampled[..., :m].reshape(x_shape)
    y_new = z_sampled[..., m:].reshape(y_shape)
    return x_new, y_new


print("CEMS functions defined.")

## 11. Neighbourhood Utilities & TwoNN
precompute_knn for anchor-based CEMS; in-house TwoNN for global ID estimation.

> **Note — augmented twins as neighbours:** With 4x flip augmentation, `precompute_knn` operates on a 1428×384 feature matrix. The 4 orientations of any given image are genuinely close in feature space and share identical Y values, so they frequently end up as each other's nearest neighbours. This is intentional and correct — they are valid CEMS neighbourhood members.

In [ ]:
def _next_power_of_2(n):
    if n < 1:
        return 1
    return 1 << (n - 1).bit_length()


def compute_neigh_size(d):
    """neigh_size = next_power_of_2(d + d*(d+1)/2 + 1)."""
    base = d + d * (d + 1) // 2
    return _next_power_of_2(base + 1)


def precompute_knn(X, Y, neigh_size, compute_device="cpu"):
    """Pre-sort training points by Euclidean distance in joint [X, Y] space.

    Returns int64 array (N, neigh_size); col 0 = self-index.
    """
    XY = np.concatenate([X, Y], axis=1).astype(np.float32)
    t  = torch.tensor(XY, dtype=torch.float32)
    if compute_device in ("cuda", "mps"):
        try:
            t = t.to(compute_device)
        except Exception:
            pass  # fall back to CPU silently

    dists = torch.cdist(t, t, p=2, compute_mode="donot_use_mm_for_euclid_dist")
    dists.fill_diagonal_(-float('inf'))  # self sorts to col 0
    sorted_idx = torch.argsort(dists, dim=1).cpu().numpy()
    return sorted_idx[:, :neigh_size].astype(np.int64)


def twonn_inhouse(X):
    """In-house TwoNN (Facco et al. 2017). Returns float ID estimate."""
    nn_model = NearestNeighbors(n_neighbors=3, algorithm="auto", metric="euclidean")
    nn_model.fit(X)
    dists, _ = nn_model.kneighbors(X)
    r1, r2  = dists[:, 1], dists[:, 2]  # col 0 is self
    mask    = r1 > 0
    r1, r2  = r1[mask], r2[mask]
    mu      = r2 / r1
    mu_s    = np.sort(mu)
    n       = len(mu_s)
    ecdf    = np.arange(1, n + 1) / n
    log_mu  = np.log(mu_s)
    log_surv = -np.log1p(-ecdf + 1e-10)
    return float(np.dot(log_mu, log_surv) / np.dot(log_mu, log_mu))


print("Neighbourhood utilities defined.")

## 12. Loss, Metrics & Dataset
Weighted SmoothL1, weighted global R², per-target RMSE, and a minimal PyTorch Dataset.

In [ ]:
WEIGHT_VECTOR = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
_LOSS_WEIGHTS = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)


def _weighted_smooth_l1(pred, target, weights, beta=1.0):
    """Per-target weighted SmoothL1, averaged over the batch."""
    loss_per = nn.functional.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss_per * weights.to(pred.device)).mean()


def weighted_global_r2(y_true, y_pred):
    """Kaggle competition metric: weighted global R² across all (image, target) pairs."""
    w  = WEIGHT_VECTOR
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


class BiomassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


@torch.no_grad()
def eval_epoch(model, loader, eval_device):
    model.eval()
    total_loss, all_pred, all_true = 0.0, [], []
    for X, y in loader:
        X, y = X.to(eval_device), y.to(eval_device)
        pred = model(X)
        total_loss += _weighted_smooth_l1(pred, y, _LOSS_WEIGHTS).item() * len(X)
        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    r2   = weighted_global_r2(y_true, y_pred)
    rmse = rmse_per_target(y_true, y_pred)
    return total_loss / len(loader.dataset), r2, rmse, y_pred, y_true


print("Loss, metrics, dataset, eval_epoch defined.")

## 13. Model Training — CEMS-L
Precompute kNN, then run anchor-based CEMS training; best checkpoint tracked by val R².

> **Epoch count — Option A (chosen):** Epochs unchanged at 80. With N_train ~1142 (vs ~286 previously), each epoch processes ~4× more anchors, so total gradient steps scale naturally with dataset size. No reduction applied.

In [ ]:
# ---- Neighbourhood precomputation ----
d_for_neigh = cfg.initial_d  # 5, from local Part-2 run (d≈4.71)
neigh_size  = compute_neigh_size(d_for_neigh)
print(f"d={d_for_neigh}  d+d*(d+1)/2={d_for_neigh + d_for_neigh*(d_for_neigh+1)//2}  neigh_size={neigh_size}")

knn_indices = precompute_knn(X_train, Y_train_scaled, neigh_size, compute_device=device)
print(f"knn_indices shape: {knn_indices.shape}")

# ---- Build datasets and loaders ----
train_ds = BiomassDataset(X_train, Y_train_raw)
val_ds   = BiomassDataset(X_val,   Y_val_raw)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

# ---- CEMS args (CEMS-L: use_hessian=False) ----
cems_args = SimpleNamespace(
    sigma=cfg.sigma,
    cems_method=cfg.cems_method,
    id=d_for_neigh,
    use_hessian=cfg.use_hessian,
)

# ---- Instantiate model ----
torch.manual_seed(cfg.seed)
model = BiomassModel(cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout).to(device)

# ---- Training function (inlined from train_cems.py + shared/train.py) ----

def train_cems_loop(model, X_train_np, Y_train_raw_np, val_ds, knn_idx, scaler, args,
                    neigh_sz, epochs, lr, weight_decay, seed, train_device, verbose=True):
    """Anchor-based CEMS training loop. Returns (history, best_state_dict)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    Y_sc = scaler.transform(Y_train_raw_np).astype(np.float32)
    X_t      = torch.tensor(X_train_np,    dtype=torch.float32, device=train_device)
    Y_raw_t  = torch.tensor(Y_train_raw_np, dtype=torch.float32, device=train_device)
    Y_sc_t   = torch.tensor(Y_sc,          dtype=torch.float32, device=train_device)

    val_ldr   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)

    N_train    = len(X_train_np)
    samples_idx = np.arange(N_train)
    history = {k: [] for k in ["train_loss", "train_loss_real", "train_loss_aug",
                                "val_loss", "val_r2", "val_rmse"]}
    best_val_r2 = -float("inf")
    best_state  = None

    mode_str = "CEMS-full" if args.use_hessian else "CEMS-L (linear)"
    if verbose:
        print(f"  mode={mode_str}  neigh_size={neigh_sz}  sigma={args.sigma}  epochs={epochs}")

    for epoch in range(1, epochs + 1):
        model.train()
        shuffle_idx  = np.random.permutation(samples_idx)
        idx_to_del   = []
        ep_loss = ep_real = ep_aug = n_steps = 0

        for anchor in shuffle_idx:
            anchor = int(anchor)
            if anchor in idx_to_del:
                continue

            neigh_row = knn_idx[anchor]
            candidates = neigh_row[1:]
            available  = candidates[~np.isin(candidates, idx_to_del)]
            n_neigh    = min(neigh_sz - 1, len(available))
            if n_neigh == 0:
                available = candidates[:neigh_sz - 1]
                n_neigh   = len(available)

            idx_all   = np.concatenate([[anchor], available[:n_neigh]])
            X_b       = X_t[idx_all]
            Y_raw_b   = Y_raw_t[idx_all]
            Y_sc_b    = Y_sc_t[idx_all]

            optimizer.zero_grad()

            # Real path
            pred_real = model(X_b)
            loss_real = _weighted_smooth_l1(pred_real, Y_raw_b, _LOSS_WEIGHTS)

            # CEMS augmentation path
            pred_aug, y_aug_sc = model.forward_cems(args, X_b, Y_sc_b)
            y_aug_np  = scaler.inverse_transform(
                y_aug_sc.detach().cpu().numpy()
            ).astype(np.float32)
            y_aug_raw = torch.tensor(y_aug_np, dtype=torch.float32, device=train_device)
            loss_aug  = _weighted_smooth_l1(pred_aug, y_aug_raw, _LOSS_WEIGHTS)

            loss = loss_real + loss_aug
            loss.backward()
            optimizer.step()

            b        = len(idx_all)
            ep_loss += loss.item() * b
            ep_real += loss_real.item() * b
            ep_aug  += loss_aug.item() * b
            n_steps += b
            idx_to_del.append(anchor)

        scheduler.step()

        tr = ep_loss / max(n_steps, 1)
        tr_r = ep_real / max(n_steps, 1)
        tr_a = ep_aug  / max(n_steps, 1)
        val_loss, val_r2, val_rmse, _, _ = eval_epoch(model, val_ldr, train_device)

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(tr)
        history["train_loss_real"].append(tr_r)
        history["train_loss_aug"].append(tr_a)
        history["val_loss"].append(val_loss)
        history["val_r2"].append(val_r2)
        history["val_rmse"].append(val_rmse)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            rmse_str = "  ".join(
                f"{t.split('_')[1]}:{v:.2f}" for t, v in val_rmse.items()
            )
            print(
                f"  ep {epoch:3d}  tr={tr:.4f}  real={tr_r:.4f}  aug={tr_a:.4f}  "
                f"val_loss={val_loss:.4f}  val_R²={val_r2:.4f}  [{rmse_str}]"
            )

    return history, best_state


# ---- Run training ----
history, best_state = train_cems_loop(
    model=model,
    X_train_np=X_train,
    Y_train_raw_np=Y_train_raw,
    val_ds=val_ds,
    knn_idx=knn_indices,
    scaler=y_scaler,
    args=cems_args,
    neigh_sz=neigh_size,
    epochs=cfg.epochs,
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
    seed=cfg.seed,
    train_device=device,
    verbose=True,
)

best_epoch = int(np.argmax(history["val_r2"])) + 1
print(f"\nBest val R² = {max(history['val_r2']):.4f} at epoch {best_epoch}")

## 14. Latent ID Estimation
Estimate intrinsic dimension on the trained 32-d latents; sanity check — expected ~4–5.

In [ ]:
# Load best weights back into model for a clean estimate
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()

with torch.no_grad():
    latents_train = model.encode(
        torch.tensor(X_train, dtype=torch.float32, device=device)
    ).cpu().numpy()

d_latent = twonn_inhouse(latents_train)
print(f"Latent ID (TwoNN, 32-d space): {d_latent:.2f}")
if d_latent < 3 or d_latent > 10:
    print("  WARNING: ID outside expected 3-10 range — check encoder quality.")
else:
    print("  ID looks reasonable (expected ~4-5).")

## 15. Final Validation Metrics
Evaluate best checkpoint; flag if weighted R² < 0.75 (possible extraction mismatch).

In [ ]:
# model already has best_state loaded from cell 14
_, val_r2_best, val_rmse_best, val_preds, val_true = eval_epoch(model, val_loader, device)

print("=" * 55)
print(f"Final val weighted R²: {val_r2_best:.4f}")
print("=" * 55)
print("RMSE per target:")
for t, v in val_rmse_best.items():
    print(f"  {t:<18}: {v:.4f}")
print("=" * 55)

# Expected from local runs: ~0.80
if val_r2_best < 0.75:
    print(
        "\n  *** WARNING: val R² = {:.4f} is significantly below the local baseline"
        " (~0.80).\n"
        "      Likely causes:\n"
        "      1. Feature extraction mismatch (DINOv2 version / resize / normalisation).\n"
        "      2. Random seed / split divergence.\n"
        "      3. Kaggle environment difference (torch/transformers version)."
        .format(val_r2_best)
    )
else:
    print(f"\n  R² = {val_r2_best:.4f} — consistent with local baseline (~0.80). OK.")

## 16. Test Inference
Load best checkpoint, encode test features through encoder+head, inverse-scale predictions.

In [ ]:
# model already has best_state loaded
model.eval()

X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

with torch.no_grad():
    test_preds_scaled = model(X_test_t).cpu().numpy()

# Predictions are on raw target scale (model was trained on raw labels)
test_preds = test_preds_scaled  # already in raw gram scale
test_preds = np.clip(test_preds, 0.0, None)  # biomass cannot be negative

print(f"test_preds shape: {test_preds.shape}")
print(f"test_preds range: [{test_preds.min():.2f}, {test_preds.max():.2f}]")
print("Sample predictions (first 3 test images):")
for i in range(min(3, len(test_image_ids))):
    print(f"  {test_image_ids[i]}: {dict(zip(TARGETS, test_preds[i].round(2)))}")

## 17. Format Predictions as Submission CSV
Map (image_id, 5 targets) → long-format rows matching test.csv sample_id keys.

In [ ]:
def prepare_submission(test_csv_path, predictions, image_ids):
    """Returns long-format DataFrame with columns [sample_id, target]."""
    df_t = pd.read_csv(test_csv_path)

    pred_dict = {
        img_id: {col: float(val) for col, val in zip(TARGETS, pred_row)}
        for img_id, pred_row in zip(image_ids, predictions)
    }

    def _get_pred(row):
        img_id      = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        val = pred_dict.get(img_id, {}).get(target_name, 0.0)
        return max(0.0, val)

    df_t["target"] = df_t.apply(_get_pred, axis=1)
    return df_t[["sample_id", "target"]]


submission = prepare_submission(TEST_CSV, test_preds, test_image_ids)

out_path = os.path.join(cfg.output_dir, "submission_cems_l_flip4x.csv")
submission.to_csv(out_path, index=False)

print(f"Submission saved to: {out_path}")
print(f"Rows: {len(submission)}")
print(submission.head(10))
print("\nTarget value stats:")
print(submission["target"].describe().round(3))
print("\nWorking dir:", os.listdir(cfg.output_dir))